In [61]:
import requests
import json
import pandas as pd
import numpy as np
from pandas import json_normalize
import plotly.graph_objects as go
import plotly.express as px
import yfinance as yf

In [62]:
df = pd.read_csv("Stock_prices_from2012.06.csv", header=[0, 1], index_col=0, parse_dates=True)
df

Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2012-05-18   15.863460   10.692500   14.883401   37.897209   23.079254   
2012-05-21   16.787682   10.905500   15.223258   33.733765   23.457739   
2012-05-22   16.658773   10.766500   14.893314   30.730145   23.465614   
2012-05-23   17.065241   10.864000   15.107988   31.721437   22.953102   
2012-05-24   16.908512   10.762000   14.964213   32.742470   22.921558   
...                ...         ...         ...         ...         ...   
2026-09-10  326.570007  251.889999  332.600006  644.380005  492.440002   
2026-09-11  332.269989  256.779999  338.500000  648.030029  495.630005   
2026-09-14  333.079987  253.539993  349.390015  665.599976  505.410004   
2026-09-15  331.339996  248.419998  344.980011  670.239990  497.119995   
2026-09-16  332.410004  245.960007  342.869995  673.309998  490.299988   

Price                                              Volume               \
Ticker            NVDA         SPY        TSLA       AAPL         AMZN   
Date                                                                     
2012-05-18    0.276238  101.239059    1.837333  732292400  104634000.0   
2012-05-21    0.281040  102.979172    1.918000  631106000   71596000.0   
2012-05-22    0.277610  103.158623    2.053333  694870400   74662000.0   
2012-05-23    0.284470  103.213242    2.068000  584897600   84876000.0   
2012-05-24    0.276924  103.416122    2.018667  496230000   62822000.0   
...                ...         ...         ...        ...          ...   
2026-09-10  218.360001  757.830017  363.559998   70011900   25484800.0   
2026-09-11  218.289993  764.289978  365.440002   50716900   26724100.0   
2026-09-14  210.960007  760.880005  358.970001   39269100   34352800.0   
2026-09-15  212.169998  757.390015  356.579987   31748200   36275400.0   
2026-09-16  213.899994  754.049988  358.079987   35981000   33488500.0   

Price                                                                   \
Ticker            GOOGL         META      MSFT         NVDA        SPY   
Date                                                                     
2012-05-18  238701060.0  573576400.0  56205300  567288000.0  319615900   
2012-05-21  122892984.0  168192700.0  38787900  416260000.0  177861100   
2012-05-22  121953924.0  101786600.0  39504900  410140000.0  197531200   
2012-05-23  126996876.0   73600000.0  65171000  496000000.0  204958400   
2012-05-24   75576348.0   50237200.0  52575000  520420000.0  167357600   
...                 ...          ...       ...          ...        ...   
2026-09-10   23557600.0   21531900.0  16038800  105768000.0   42740400   
2026-09-11   24708300.0   16925000.0  14510500   89060100.0   45512700   
2026-09-14   35905400.0   19332300.0  23091900  132267200.0   43992400   
2026-09-15   21928600.0   19502800.0  17794600   88059700.0   46195000   
2026-09-16   18928900.0   17217800.0  16670300   96563600.0   59217700   

Price                   
Ticker            TSLA  
Date                    
2012-05-18  24247500.0  
2012-05-21  22128000.0  
2012-05-22  35493000.0  
2012-05-23  18306000.0  
2012-05-24  16134000.0  
...                ...  
2026-09-10  29667200.0  
2026-09-11  30153000.0  
2026-09-14  32477200.0  
2026-09-15  30317700.0  
2026-09-16  32177400.0  

[3602 rows x 16 columns]

In [63]:
df2 = df.drop(columns=["Volume"]).droplevel(0, axis=1)
df2

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-05-18,15.863460,10.692500,14.883401,37.897209,23.079254,0.276238,101.239059,1.837333
2012-05-21,16.787682,10.905500,15.223258,33.733765,23.457739,0.281040,102.979172,1.918000
2012-05-22,16.658773,10.766500,14.893314,30.730145,23.465614,0.277610,103.158623,2.053333
2012-05-23,17.065241,10.864000,15.107988,31.721437,22.953102,0.284470,103.213242,2.068000
2012-05-24,16.908512,10.762000,14.964213,32.742470,22.921558,0.276924,103.416122,2.018667
...,...,...,...,...,...,...,...,...
2026-09-10,326.570007,251.889999,332.600006,644.380005,492.440002,218.360001,757.830017,363.559998
2026-09-11,332.269989,256.779999,338.500000,648.030029,495.630005,218.289993,764.289978,365.440002
2026-09-14,333.079987,253.539993,349.390015,665.599976,505.410004,210.960007,760.880005,358.970001


In [64]:
normalized = ((df2 / df2.iloc[0])-1)*100
# Equal weight M7 average (ideal/theoretical distribution — 1/7 each)
normalized["M7"]= normalized[["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"]].mean(axis=1)
# Simulating 1000 randomly weighted M7 portfolios and averaging to approximate realistic investor returns
mag7 = ["GOOGL", "AMZN", "AAPL", "META", "MSFT", "NVDA", "TSLA"]

n_simulations = 1
results = []

for _ in range(n_simulations):
    weights = np.random.dirichlet(np.ones(7))
    portfolio_return = (normalized[mag7] * weights).sum(axis=1)
    results.append(portfolio_return)

simulations = pd.DataFrame(results).T
simulations.index = normalized.index

# One average line
normalized["IRL-M7"] = simulations.mean(axis=1)
normalized.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA,M7,IRL-M7
Date,,,,,,,,,,
2012-05-18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-05-21,5.826106,1.992053,2.283464,-10.986151,1.639935,1.738397,1.718815,4.390441,0.983464,-4.894009
2012-05-22,5.013496,0.692077,0.066607,-18.911853,1.674058,0.496666,1.896070,11.756175,0.112461,-9.483705
2012-05-23,7.575783,1.603930,1.508979,-16.296112,-0.546604,2.980074,1.950020,12.554454,1.340072,-7.908688
2012-05-24,6.587797,0.649988,0.542970,-13.601897,-0.683279,0.248322,2.150418,9.869415,0.516188,-6.781327


In [65]:
fig1 = px.line(normalized,x=normalized.index,y=["M7","SPY", "IRL-M7"])
fig1.update_layout(title=dict(text="M7 vs SPY",font=dict(size=24)),yaxis_title="Change(%)")
fig1.show()

In [66]:
fig4 = px.line(normalized,x=normalized.index,y=["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"])
fig4.update_traces(opacity=0.7)
fig4.update_layout(title=dict(text="The magnificent seven's growth over-time",font=dict(size=24)))
fig4.show()

In [67]:
# Resample to monthly to make animation smoother and faster
normalized_yearly = normalized.resample("YE").last()

# Reshape to long format
normalized_long = normalized_yearly.reset_index().melt(
    id_vars="Date",
    var_name="Company",
    value_name="Return"
)

normalized_long["Date"] = normalized_long["Date"].dt.strftime("%Y-%m")

# Animate
fig5 = px.bar(normalized_long,
              x="Company",
              y="Return",
              animation_frame="Date",
              range_y=[0.01, normalized_long["Return"].max()],
              log_y=True,
              title="companies returns from 2012"
              )
fig5.show()

In [68]:
top7_2012 = ["AAPL", "XOM", "MSFT", "IBM", "GE", "CVX", "BRK-B"]
data_2012 = yf.download(top7_2012, start="2012-05-18", end="2026-09-16")["Close"]
print(data_2012.head())

[*********************100%***********************]  6 of 7 completed


Ticker           AAPL      BRK-B        CVX         GE         IBM       MSFT  \
Date                                                                            
2012-05-18  15.863461  78.910004  54.924278  70.383804  110.360786  23.079256   
2012-05-21  16.787672  79.800003  55.610432  71.015205  111.420036  23.457739   
2012-05-22  16.658772  79.650002  55.403996  71.238060  110.890442  23.465618   
2012-05-23  17.065243  79.750000  55.225521  71.238060  110.496056  22.953096   
2012-05-24  16.908514  79.800003  55.816826  71.498062  110.479103  22.921564   

Ticker            XOM  
Date                   
2012-05-18  46.572098  
2012-05-21  46.897926  
2012-05-22  46.846485  
2012-05-23  46.897926  
2012-05-24  47.223789  


In [69]:
normalized_1= ((data_2012/data_2012.iloc[0])-1)*100
normalized_1["M7"]= normalized_1[top7_2012].mean(axis= 1)
normalized_1.head()

Ticker,AAPL,BRK-B,CVX,GE,IBM,MSFT,XOM,M7
Date,,,,,,,,
2012-05-18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-05-21,5.826040,1.127866,1.249272,0.897083,0.959806,1.639926,0.699622,1.771374
2012-05-22,5.013477,0.937774,0.873416,1.213711,0.479931,1.674066,0.589167,1.540220
2012-05-23,7.575789,1.064499,0.548469,1.213711,0.122570,-0.546637,0.699622,1.525432
2012-05-24,6.587803,1.127866,1.625051,1.583117,0.107209,-0.683263,1.399317,1.678157


In [70]:
fig6=go.Figure()
#first dataset(SPY)
fig6.add_scatter(x=normalized.index,y=normalized["SPY"],name="SPY",mode="lines")
#second dataset(past M7)
fig6.add_scatter(x=normalized_1.index,y=normalized_1["M7"],name="past M7",mode="lines")

fig6.show()


In [71]:
print("Current M7:", normalized["M7"].iloc[-1].round(2), "%")
print("2012 Top 7:", normalized_1["M7"].iloc[-1].round(2), "%")
print("SPY:", normalized["SPY"].iloc[-1].round(2), "%")

Current M7: 15260.43 %
2012 Top 7: 802.7 %
SPY: 644.82 %


## Part 3

In [72]:
individual_growth={}
for ticker in mag7:
    data=yf.download(ticker,period="max")["Close"].squeeze().dropna()
    normalized_ticker= ((data/data.iloc[0])-1 )*100
    individual_growth[ticker]= normalized_ticker
df_individual=pd.DataFrame(individual_growth)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [73]:
final_returns = df_individual.iloc[-1]

# Calculate years public
years = df_individual.apply(lambda x: x.dropna().shape[0] / 252).round(1)

fig7 = px.bar(x=final_returns.index, y=final_returns.values,
              title="Total Return Since IPO",
              labels={"x": "Company", "y": "% Return"},
              log_y=True,
              text=[f"{y} yrs" for y in years])

fig7.update_traces(textposition="outside")
fig7.show()

## Part 4

In [74]:
growth_velocity = ((1 + final_returns/100) ** (1/years) - 1) * 100
fig_velocity=px.bar(growth_velocity,x=growth_velocity.index,y=growth_velocity.values)
fig_velocity.show()

# Dan's work

In [75]:
change = df["Close"].pct_change()*100
change

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-05-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-05-21,5.826106,1.992053,2.283464,-10.986151,1.639935,1.738397,1.718815,4.390441
2012-05-22,-0.767873,-1.274586,-2.167366,-8.903898,0.033573,-1.220514,0.174259,7.055947
2012-05-23,2.439960,0.905585,1.441412,3.225800,-2.184099,2.471135,0.052946,0.714304
2012-05-24,-0.918409,-0.938883,-0.951649,3.218745,-0.137427,-2.652699,0.196565,-2.385546
...,...,...,...,...,...,...,...,...
2026-09-10,3.561239,-0.202058,0.589751,-1.424222,0.160685,-2.264792,-0.599424,-1.155488
2026-09-11,1.745409,1.941323,1.773901,0.566440,0.647795,-0.032061,0.852429,0.517110
2026-09-14,0.243777,-1.261783,3.217139,2.711286,1.973246,-3.357912,-0.446162,-1.770469


In [76]:
change_sum = ((1 + change / 100).cumprod() - 1) * 100
change_sum



Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-05-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-05-21,5.826106,1.992053,2.283464,-10.986151,1.639935,1.738397,1.718815,4.390441
2012-05-22,5.013496,0.692077,0.066607,-18.911853,1.674058,0.496666,1.896070,11.756175
2012-05-23,7.575783,1.603930,1.508979,-16.296112,-0.546604,2.980074,1.950020,12.554454
2012-05-24,6.587797,0.649988,0.542970,-13.601897,-0.683279,0.248322,2.150418,9.869415
...,...,...,...,...,...,...,...,...
2026-09-10,1958.630436,2255.763355,2134.704339,1600.336302,2033.691146,78947.881793,648.554976,19687.376850
2026-09-11,1994.561953,2301.496339,2174.345775,1609.967682,2047.513094,78922.538641,654.935874,19789.699342
2026-09-14,1999.668012,2271.194675,2247.514635,1656.329794,2089.888808,76269.031083,651.567635,19437.558421


In [77]:
figD1 = px.line(change_sum, color='Ticker')

figD1.update_layout(title={"text":"Percentage change of stocks over time", 'x':0.5, 'xanchor':'center'}, font=dict(size=16), 
        yaxis={"title": {"text": "Change %", "font": {"size": 14}}},
        xaxis={
            "title": {"text": "Date", "font": {"size": 14}}},
        height=600)
figD1.show()

# Comparing Normalization vs percent-change


## This is a nice tool to filter from certain dates. 
if the date is not in the list, the next valid date is used.

In [89]:
def rebase(change, start, end):
    return ((1 + change.loc[start:end] / 100).cumprod() - 1) * 100

start_date = str(input())
end_date = str(input())
rebase(change, start_date, end_date)


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,-2.897522,-2.202808,-1.700915,-6.351350,-2.535148,-3.620291,-2.517687,-4.576272
2012-06-04,-2.326322,0.779674,-0.390801,-9.121636,-2.192547,-5.631478,-2.563320,-5.491526
2012-06-05,-2.579062,0.140899,-1.799050,-12.601354,-2.329589,-2.896184,-1.825495,-5.389833
2012-06-06,-1.085276,2.221594,-0.049912,-9.425669,0.548151,-0.321770,0.380321,-0.949174
2012-06-07,-1.040274,2.766421,-0.452761,-11.114858,0.137009,-4.344271,0.441147,-1.932208
...,...,...,...,...,...,...,...,...
2026-09-09,1724.916798,2270.954765,2196.337065,2127.805441,2036.105784,78502.281739,643.159589,18602.199562
2026-09-10,1789.906447,2266.164060,2209.879739,2096.076543,2039.538189,76722.103792,638.704914,18386.097910
2026-09-11,1822.893040,2312.098956,2250.854713,2108.515993,2053.398014,76697.474236,645.001847,18481.691360


## It can be immedieately called in for a plot. It sort of works as a time range checker.

In [91]:
rebased = rebase(change, start_date, end_date)
close = df["Close"].loc[rebased.index]
daily  = change.loc[rebased.index]

figD2 = px.line(rebased, color='Ticker',
               labels={"value": "Change %", "index": "Date"})

for tr in figD2.data:
    tr.customdata = np.stack([
        daily[tr.name].to_numpy(),
        close[tr.name].to_numpy(),],
        axis = 1)
    tr.hovertemplate = (
        "<b>%{fullData.name}</b><br>"
        "Total Change: %{y:.2f}%<br>"
        "Daily Change: %{customdata[0]:.2f}%<br>" 
        "Close Price: $%{customdata[1]:.2f}<br>"
        "Date: %{x|%d %b %Y}<br>"
        "<extra></extra>"
    )

figD2.update_layout(
    title={"text": "Percentage change of stocks over time", 'x': 0.5, 'xanchor': 'center'},
    font=dict(size=16),
    yaxis={"title": {"text": "Change %", "font": {"size": 18}}},
    xaxis={"title": {"text": "Date", "font": {"size": 18}}, "hoverformat": "%d %b %Y"},
    height=600,
)
figD2.show()

## Lets check when did the market fell 10 or more %
as a follow up would be interesting to check the positive as well.

In [80]:
spy = df["Close"]["SPY"]
dd = (spy / spy.cummax() - 1) * 100      # % below the highest point so far
dd

Date
2012-05-18    0.000000
2012-05-21    0.000000
2012-05-22    0.000000
2012-05-23    0.000000
2012-05-24    0.000000
                ...   
2026-09-10   -2.577517
2026-09-11   -1.747060
2026-09-14   -2.185427
2026-09-15   -2.634081
2026-09-16   -3.063457
Name: SPY, Length: 3602, dtype: float64

In [81]:
figD3 = px.line(dd,)

figD3.update_traces(hovertemplate = (
        "<b>SnP 500</b><br>"
        "Change since last max = %{y:.2f}%<br>" 
        "Date: %{x|%d %b %Y}<br>"
        "<extra></extra>"))

figD3.update_layout(
    title={"text": "Percentage fall (drawdown) compared to previous values represented on the SnP 500", 'x': 0.5, 'xanchor': 'center'},
    font=dict(size=16),
    yaxis={"title": {"text": "Change %", "font": {"size": 16}}},
    xaxis={"title": {"text": "Date", "font": {"size": 16}}, "hoverformat": "%d %b %Y"},
    showlegend=False,
    height=600,
)


figD3.add_annotation(
    x="2015-08-25", y=-11.9,
    text="Panic over Chinese economy",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=-150, ay=-20,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)

figD3.add_annotation(
    x="2016-02-11", y=-13.0,
    text="Fear of slowing economy",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=0, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


figD3.add_annotation(
    x="2018-12-24", y=-19.4,
    text="US political tension",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=-100, ay=20,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


figD3.add_annotation(
    x="2020-03-23", y=-33.7,
    text="COVID crash",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,arrowcolor="crimson",
    ax=-100, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)

figD3.add_annotation(
    x="2025-4-08", y=-18.8,
    text="Chinese trade tension",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=50, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


figD3.add_vrect(
    x0="2022-01-03", x1="2022-10-12",
    fillcolor="red", opacity=0.12, line_width=0,
    annotation_text="Fed hiking cycle", annotation_position="bottom",
)


figD3.show()

## Investment calculator - we shoudl combine it somehow with selecting specific stocks 

In [82]:

#select a time and date between 2012-06 and 2026-09-15 for start and end
def yield_calc(change, start, end, invest):
    growth = (1 + change["SPY"].loc[start:end] / 100).prod()
    return invest * growth

y_start_date = (input())
y_end_date = (input())
amount = float(input())
yield_calc(change, y_start_date, y_end_date, amount)
print(f"By investing {amount:,.0f}€ from {y_start_date} until {y_end_date} would result you {yield_calc(change, y_start_date, y_end_date, amount):,.2f}€, which is {yield_calc(change, y_start_date, y_end_date, amount)-amount:,.2f}€ profit and it equals to {yield_calc(change, y_start_date, y_end_date, amount)/amount*100:,.2f}% yield")


By investing 10,000€ from 2015 until 2025 would result you 39,952.91€, which is 29,952.91€ profit and it equals to 399.53% yield


## Future projext - Would be interesting to calculate what happens if you try to time the market.


# Trading volumes vs price or price change


In [83]:
change_vol = df["Volume"].pct_change()*100
change_vol


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-05-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-05-21,-13.817759,-31.574823,-48.515945,-70.676496,-30.988893,-26.622809,-44.351611,-8.741107
2012-05-22,10.103596,4.282362,-0.764128,-39.482154,1.848515,-1.470235,11.059248,60.398590
2012-05-23,-15.826376,13.680319,4.135129,-27.691857,64.969409,20.934315,3.760014,-48.423633
2012-05-24,-15.159508,-25.983788,-40.489601,-31.742935,-19.327615,4.923387,-18.345576,-11.864962
...,...,...,...,...,...,...,...,...
2026-09-10,6.660420,-22.902996,-28.939780,-40.023732,24.432100,27.499684,30.256854,-8.896552
2026-09-11,-27.559601,4.862899,4.884623,-21.395697,-9.528768,-15.796744,6.486369,1.637499
2026-09-14,-22.571963,28.546144,45.317161,14.223338,59.139244,48.514542,-3.340386,7.708022


In [86]:
from plotly.subplots import make_subplots
price = df["Close"]
vol = df["Volume"]

tickers=list(price.columns)

figD4 = make_subplots(rows=3, cols=3, subplot_titles = tickers, horizontal_spacing=0.07, vertical_spacing=0.09)

for i, t in enumerate(tickers):
    r, c = divmod(i, 3)
    x = price[t]
    y = vol[t]

    ok = x.notna() & y.notna() & (y > 0)
    logy = np.log10(y[ok])
    slope, intercept = np.polyfit(x[ok], logy, 1)
    corr = np.corrcoef(x[ok], logy)[0, 1]

    xs = np.linspace(x[ok].min(), x[ok].max(), 100)
    ys = 10 ** (slope * xs + intercept)

    figD4.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=3, opacity=0.3),
            showlegend=False,
        ),
        row=r + 1, col=c + 1,
    )

    figD4.add_trace(
        go.Scatter(x=xs, y=ys, mode="lines",
                   line=dict(color="red", width=2),
                   showlegend=False, hoverinfo="skip"),
        row=r + 1, col=c + 1,
    )

    figD4.add_annotation(
        text=f"R²={corr**2:.2f}", x=0.05, y=0.95,
        xref="x domain", yref="y domain", showarrow=False,
        font=dict(size=12, color="red"),
        row=r + 1, col=c + 1,
    )

figD4.update_layout(height=900, title={"text":"Traded volume vs price", 'x':0.5, 'xanchor':'center'}, font=dict(size=16),
    hovermode=False)
figD4.update_yaxes(type="log")
figD4.show()

# Checking the same sort of trend but with the percentage change compared to volume change


In [47]:
rel_vol = vol / vol.rolling(63).median()

long = (change.stack().rename("change").to_frame()
        .join(rel_vol.stack().rename("rel_vol"))
        .join(vol.stack().rename("volume"))
        .dropna().reset_index())

surge = long[long["rel_vol"] >= 2.5].copy()     # days with 2x+ normal volume

figD5 = px.scatter(
    surge, x="Date", y="change",
    size="rel_vol", color="rel_vol",
    color_continuous_scale="Viridis",
    facet_col="Ticker", facet_col_wrap=2,
    size_max=18, opacity=0.7, height=1000,
    labels={"change": "Daily change (%)", "rel_vol": "Volume / 63d median"},
    hover_data={"Date": "|%d %b %Y", "change": ":.2f", "rel_vol": ":.1f", "volume": ":,.0f"},
    )
figD5.add_hline(y=0, line_dash="dash", line_color="grey")
figD5.update_layout(title={"text":"High-volume days: when they happened, which way price moved, how big the surge", 'x':0.5, 'xanchor':'center'}, font=dict(size=16)),

figD5.show()